# ego2robot: train SmolVLA on robot data generated from my phone demos

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells one by one (Shift+Enter).

Pipeline: 20 phone demos → retargeted to a Panda in simulation (`sim/replay_demos.py --save`, run on the Mac)
→ LeRobot dataset → fine-tune `lerobot/smolvla_base` → evaluate on 50 unseen random scenes.

## 1. Settings

In [ ]:
GITHUB_USER = "acristinaa"
BRANCH = "train-smolvla"
EPISODES_ZIP = "/content/drive/MyDrive/ego2robot/sim_episodes.zip"
WORK = "/content/drive/MyDrive/ego2robot"
STEPS, BATCH = 5000, 16
import os

## 2. Connect Google Drive and download the code

In [ ]:
%cd /content
from google.colab import drive
drive.mount("/content/drive")
!rm -rf /content/ego2robot && git clone -q -b {BRANCH} https://github.com/{GITHUB_USER}/ego2robot.git /content/ego2robot
%cd /content/ego2robot
!git log --oneline -3

## 3. Install LeRobot (SmolVLA) and the simulator

In [ ]:
!pip install -q "lerobot[smolvla,dataset]==0.6.1" "robosuite==1.4.0" "mujoco==3.8.1" termcolor numba "imageio[ffmpeg]"
import os, lerobot
# SmolVLA hard-codes bfloat16 for the VLM. The T4 has no native bf16, so it is emulated (~7 s/step)
# and fp16 mixed precision crashes on bf16 gradients. Load in fp32 and let AMP use the fp16 tensor cores.
LR = os.path.dirname(lerobot.__file__)
!sed -i 's/torch_dtype="bfloat16"/torch_dtype="float32"/' {LR}/policies/smolvla/smolvlm_with_expert.py
os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"
!python -c "import lerobot, robosuite, torch; print('lerobot', lerobot.__version__, '| robosuite', robosuite.__version__, '| cuda', torch.cuda.is_available())"

## 4. Unpack the simulated episodes and convert them to a LeRobot dataset

In [ ]:
DS = "/content/ego2robot_ds"
DS_DRIVE = f"{WORK}/dataset"
if os.path.exists(f"{DS_DRIVE}/DONE"):
    print("Dataset already converted earlier - copying it from Drive")
    !rm -rf {DS} && cp -r {DS_DRIVE} {DS}
else:
    !mkdir -p /content/sim_episodes && unzip -q -o {EPISODES_ZIP} -d /content/sim_episodes
    EP_DIR = "/content/sim_episodes/sim_episodes"
    !ls {EP_DIR} | grep -c "npz$"
    !rm -rf {DS} && python -m sim.to_lerobot {EP_DIR} --repo-id local/ego2robot_bowl_on_plate --root {DS} 2>&1 | grep -v "moov atom" | tail -3
    !rm -rf {DS_DRIVE} && cp -r {DS} {DS_DRIVE} && touch {DS_DRIVE}/DONE
!ls {DS}

## 5. Fine-tune SmolVLA

Checkpoints are written to Google Drive every 500 steps. If Colab disconnects, run cells 1-4 again
(cell 4 then just copies the dataset back from Drive), set `RESUME_FROM` to the newest checkpoint folder
(e.g. `"002000"`) and run this cell again. Google Drive turns the `last` shortcut into a plain file,
so the numbered folder is used instead.

`STEPS` is the total: resuming from `002000` with `STEPS = 5000` trains 3000 more steps.

In [ ]:
RESUME_FROM = None
OUT = f"{WORK}/train/smolvla"
if RESUME_FROM:
    cmd = " ".join([
        "lerobot-train",
        f"--config_path={OUT}/checkpoints/{RESUME_FROM}/pretrained_model/train_config.json",
        "--resume=true", f"--steps={STEPS}",
    ])
else:
    if os.path.exists(f"{OUT}/checkpoints"):
        raise RuntimeError(f"{OUT} already has checkpoints. Set RESUME_FROM, or delete the folder yourself.")
    cmd = " ".join([
        "lerobot-train",
        "--policy.path=lerobot/smolvla_base",
        "--policy.push_to_hub=false",
        "--policy.device=cuda",
        "--dataset.repo_id=local/ego2robot_bowl_on_plate",
        f"--dataset.root={DS}",
        f"--batch_size={BATCH}", f"--steps={STEPS}",
        "--save_freq=500", "--log_freq=50", "--num_workers=2",
        "--policy.use_amp=true",
        f"--output_dir={OUT}", "--job_name=smolvla_ego2robot",
        "--wandb.enable=false",
    ])
print(cmd)
!{cmd}

## 6. Evaluate on 50 random scenes the policy has never seen

In [ ]:
CKPT_STEP = "005000"
CKPT = f"{WORK}/train/smolvla/checkpoints/{CKPT_STEP}/pretrained_model"
!python -m sim.eval_policy {CKPT} --episodes 50 --videos 6 --out {WORK}/eval_{CKPT_STEP} 2>&1 | grep -v WARNING

Results and the first 6 videos are now in Google Drive → `ego2robot/eval_smolvla/`.